# Phase 10 — District Production Context (v1)

A **descriptive**, leakage-safe district production context from APY for Onion, Potato and Wheat: contract `phase10-production-context-v1` with the PQ-1 v1 quality flags. **No production-forecasting model** is built (D-46). The phase is still named "Supply Prediction" in the roadmap, because the renaming decision D-49 is not accepted.

**This notebook is read-only.**
- It verifies the frozen outputs in `Processed_Data/production_context/` and explains them. It does not rebuild them, freeze anything or write any file.
- Rows are shown only for the **design window** (source years ≤ 2018-19), rebuilt in memory with the frozen code.
- The reserved source years 2019-20 … 2021-22 appear **only as counts** taken from the output manifest. Their values are never read or displayed here.

## INDEX
1. Objective and accepted decisions
2. Data contract and provenance
3. Input and output integrity
4. Design-window rows and quality classes
5. Annual comparability and class changes
6. R4 diagnostic (diagnostic only)
7. R13: provider query pending (no cause asserted)
8. Reserved years: counts only
9. Limitations and downstream implications

## 1. Objective and accepted decisions

In [1]:
import os, sys, re
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)

from Code.production import pipeline as pc
from Code.integration.pipeline import file_sha256, frame_fingerprint

OUT = pc.OUT_DIR
log_text = pc.DECISION_LOG.read_text(encoding="utf-8")
print("D-46, D-47, D-48 recorded as ACCEPTED:", pc.decisions_accepted())
for d in pc.DECISIONS + ["D-49"]:
    m = re.search(rf"^### {d} — .*$", log_text, re.M)
    print(m.group(0)[:150] if m else f"{d}: not in the decision log (proposal only, in the Phase 10 plan)")

D-46, D-47, D-48 recorded as ACCEPTED: True
### D-46 — [ACCEPTED 2026-10-23 by the project owner; proposed earlier the same day] Phase 10 scope: descriptive district production context, contract
### D-47 — [ACCEPTED 2026-10-23 by the project owner; proposed earlier the same day] APY information availability (`apy_lag2_policy_v1`) and the Phase
### D-48 — [ACCEPTED 2026-10-23 by the project owner; proposed earlier the same day] PQ-1 v1 production quality rules (flags only; no numerical thresh
D-49: not in the decision log (proposal only, in the Phase 10 plan)


**Accepted on 2026-10-23** (full text in `Documentation/Decision_Log.md`):
- **D-46:** Option A (descriptive context, no forecasting model). The v1 contract keeps the four comparability classes separate, defers the 3-year trailing summary to v1.1, and authorises **no** procurement cap or marketable-surplus estimate.
- **D-47:** `apy_lag2_policy_v1`. APY year Y1–Y2 is usable from 1 July of Y2 + 1, so decision year t uses source year t − 2. This is a **policy assumption, not a verified publication schedule**, and it is not shown to be conservative for source years ≥ 2019-20. The design window is ≤ 2018-19; 2019-20 … 2021-22 are reserved; 2022-23 is excluded.
- **D-48:** PQ-1 v1 only flags, with no numerical threshold. It includes the R8a and R10 clarifications and R13 Option 1. Blocked rules stay inactive.

D-49 (phase names) is **not** accepted.

## 2. Data contract and provenance

In [2]:
print("contract:", pc.CONTRACT_VERSION, "| quality rules:", pc.PQ_VERSION, "| information rule:", pc.INFORMATION_RULE)
print("grain: one row per", pc.CONTEXT_KEY)
print("columns:", pc.CONTRACT_COLUMNS)
print("design window: source years <=", pc.DESIGN_END, "| reserved:", pc.RESERVED, "| excluded: 2022-2023 onward")
prov = pd.read_csv(OUT / "pq1_provenance.csv")
prov[["code", "name", "status", "effect", "active", "condition"]]

contract: phase10-production-context-v1 | quality rules: pq1-v1 | information rule: apy_lag2_policy_v1
grain: one row per ['crop', 'apy_state', 'apy_district', 'decision_agri_year']
columns: ['crop', 'apy_state', 'apy_district', 'apy_district_code', 'decision_agri_year', 'source_apy_year', 'available_from', 'information_rule', 'production_tonnes', 'area_ha', 'yield_t_per_ha', 'annual_method', 'seasons_reported', 'annual_comparability', 'pq_class', 'pq_reasons', 'pq_version', 'contract_version']
design window: source years <= 2018-2019 | reserved: ('2019-2020', '2020-2021', '2021-2022') | excluded: 2022-2023 onward


,code,name,status,effect,active,condition
0,R1,missing_value,specified,unusable,True,"production_tonnes null, or area_ha null, or annual_method == ambiguous_multi_season_no_total"
1,R2,yield_zero_encodes_missing,specified,annotated,True,production null and published yield == 0; the contract yield is null
2,R3,zero_production,review-only,review,True,production == 0 and area > 0
3,R8a,first_reported_after_state_start,specified,annotated,True,"first source year of (crop, state, district) > first source year of (crop, state); earlier years only"
4,R9,single_season_annual,specified,annotated,True,annual_comparability == single_named_season (all crops)
5,R10,method_change,specified,annotated,True,"annual_comparability differs from the most recent earlier source year with a row for the same crop, state and district"
6,R11,unit_label,specified,unusable,True,a raw season row has unit labels other than Hectare / Tonnes / Tonnes/Hectare
7,R12,duplicate_conflict,specified,unusable,True,"a raw season row key (crop, state, district, year, season) is duplicated"
8,R13,provider_query_pending,review-only,review,True,"crop Potato, source year 1997-98, state West Bengal or Gujarat (D-48 Option 1); no cause asserted"
9,R4,yield_inconsistent,diagnostic,none,False,"manifest only, per crop: rows with area > 0 and production > 0 (non-null), and max |yield - production/area| / (prod..."


**Provenance.**
- `apy_annual.csv` is the Phase 4 D-20 annual table. Values are used **as published**; the only change is that yield is set to null where production is null (the source encodes missing production with a yield of 0).
- The raw APY file supplies the unit labels (R11) and the season keys (R12).
- `apy_season.csv` is read by the audit, and the D-18 crosswalk is fingerprint-checked only.
- **`pq_class` precedence:** unusable (R1, R11, R12) > review (R3, R13) > annotated (R2, R8a, R9, R10) > ok.
- **Blocked:** the R4 row flag, R5, R6, R7 and R8b produce no flag. Activating one needs a new decision and a new PQ version.

## 3. Input and output integrity

In [3]:
spec = pc.default_spec()
inputs = pc.verify_inputs(spec["paths"], spec["records"], spec["git_tracked"])   # raises on any mismatch
inputs[["input", "path", "check", "status"]]

,input,path,check,status
0,annual,Processed_Data/integration/apy_annual.csv,file_sha256,match
1,annual,Processed_Data/integration/apy_annual.csv,shape,match
2,annual,Processed_Data/integration/apy_annual.csv,content_sha256,match
3,season,Processed_Data/integration/apy_season.csv,shape,match
4,season,Processed_Data/integration/apy_season.csv,content_sha256,match
5,crosswalk,Processed_Data/integration/district_crosswalk.csv,shape,match
6,crosswalk,Processed_Data/integration/district_crosswalk.csv,content_sha256,match_after_alias_basis_canonicalisation
7,raw,Dataset/crop-wise-area-production-yield.csv,file_sha256,match
8,crosswalk,Processed_Data/integration/district_crosswalk.csv,git_blob_vs_HEAD,match


In [4]:
# The crosswalk: Phase 4 hashed the in-memory frame, where alias_basis held ""; CSV readback turns "" into NaN.
# Only alias_basis is canonicalised (NaN -> ""), only for this comparison; the file and the record stay unchanged.
inputs.loc[inputs["check"] == "content_sha256", ["input", "recorded", "recomputed", "canonicalised", "status"]]

,input,recorded,recomputed,canonicalised,status
2,annual,1a48f31da03271db778567e94955ea41f0013c55b12723cf0d05cb0ead4bab8c,1a48f31da03271db778567e94955ea41f0013c55b12723cf0d05cb0ead4bab8c,,match
4,season,ef8f13f4a42d120a320aada2dfa6c062aa872e6a3026e5c88a07b86be1e16f9b,ef8f13f4a42d120a320aada2dfa6c062aa872e6a3026e5c88a07b86be1e16f9b,,match
6,crosswalk,a6d33dc835cb1661edfb5877d4948efd8c563dffe3c1ca7c36f8c9ab469d5b65,e5aa515275509181514b20b6eb9ff3ec9410a88af8f496055e68cc9c0ec02ded,a6d33dc835cb1661edfb5877d4948efd8c563dffe3c1ca7c36f8c9ab469d5b65,match_after_alias_basis_canonicalisation


In [5]:
pc.verify_freeze(spec)    # code, accepted decision texts, inputs, rule table and design rows still equal the freeze
man = pd.read_csv(OUT / "output_manifest.csv", dtype=str)
outs = man[man["item"] == "output"].copy()
outs["file_sha256_now"] = [file_sha256(OUT / n) for n in outs["name"]]
outs["recorded_matches"] = [f"file_sha256={h}" in v for h, v in zip(outs["file_sha256_now"], outs["value"])]
print("freeze verified; freeze manifest hash recorded in output manifest:",
      man.set_index("name").loc["freeze_manifest_sha256", "value"] == file_sha256(OUT / "pq1_freeze_manifest.csv"))
outs[["name", "recorded_matches"]]

freeze verified; freeze manifest hash recorded in output manifest: True


,name,recorded_matches
17,production_context_v1.csv,True
18,pq1_provenance.csv,True


The freeze (`pq1_freeze_manifest.csv`) was written once, **before** the reserved years were built. The full build refuses to run unless the freeze verifies and the design-window rows come out identical, which shows there is no look-ahead. Rebuilding reproduces every output byte for byte.

**Reproduce from the project root:**
1. `C:\Python313\python.exe -B -m Code.production.apy_audit`
2. `… -m Code.production.pipeline design`
3. `… -m Code.production.pipeline build`

The freeze stage runs once only, and has already been run.

## 4. Design-window rows and quality classes

In [6]:
annual, raw = pc.load(spec["paths"], pc.DESIGN_END)          # cut to source years <= 2018-19 on load
design = pc.build_context(annual, raw)
frozen = pd.read_csv(OUT / "pq1_freeze_manifest.csv", dtype=str).set_index(["item", "name"])["value"]
assert design["source_apy_year"].max() == pc.DESIGN_END
print("design rows:", len(design), "| equal to the frozen design rows:",
      frame_fingerprint(design) == frozen[("design", "content_sha256")])
pd.crosstab(design["crop"], design["pq_class"], margins=True)

design rows: 27531 | equal to the frozen design rows: True


pq_class,annotated,ok,review,unusable,All
crop,,,,,
Onion,3027,4871,5,42,7945
Potato,3834,4991,41,35,8901
Wheat,10654,5,3,23,10685
All,17515,9867,49,100,27531


In [7]:
reasons = design.assign(rule=design["pq_reasons"].str.split("|")).explode("rule")
reasons = reasons[reasons["rule"] != ""]
pd.crosstab(reasons["rule"], reasons["crop"], margins=True)

crop,Onion,Potato,Wheat,All
rule,,,,
first_reported_after_state_start,1632,1295,1761,4688
method_change,1060,858,136,2054
missing_value,42,35,23,100
provider_query_pending,0,37,0,37
single_season_annual,1239,2483,10610,14332
yield_zero_encodes_missing,42,17,23,82
zero_production,5,4,3,12
All,4020,4729,12556,21305


**Reading the classes.**
- **Almost every Wheat row is `annotated`.** Wheat is reported as a single Rabi row, and R9 annotates single named seasons for all crops. The Wheat exception was withdrawn under D-48, so this is the accepted rule working as intended, not a data problem.
- **R8a annotates every row of a district that first appears after its state's first year** (computed per crop and state). The source re-labels history with present-day geography (F15), so this is **not** evidence of a boundary split; R8b, which would identify splits, is blocked.
- **`unusable` rows** have missing values or are ambiguous (R1). Their rows are kept, never deleted.

## 5. Annual comparability and class changes

In [8]:
pd.crosstab(design["crop"], design["annual_comparability"], margins=True)

annual_comparability,ambiguous,single_named_season,verified_total,whole_year_row,All
crop,,,,,
Onion,0,1239,1824,4882,7945
Potato,18,2483,1195,5205,8901
Wheat,0,10610,19,56,10685
All,18,14332,3038,10143,27531


In [9]:
changes = pd.read_csv(OUT / "audit" / "comparability_changes.csv")
changes

,crop,districts,districts_with_class_change
0,Onion,560,297
1,Potato,581,341
2,Wheat,603,69


**Why the four classes stay separate.**
- `verified_total` is APY's Total row; it equals the sum of the seasons in 99.9% of groups (D-20).
- `whole_year_row` is a single "Whole Year" row; DES has not confirmed that it is the full-year total.
- `single_named_season` is one Rabi, Kharif or similar row, and may cover only part of the year.
- `ambiguous` means several seasons and no Total; its value is left missing (D-20).

Putting them in one column would compare unlike quantities, so a consumer must state which classes it accepts. R10 marks a change of class from the district's most recent earlier year. Within the design window, such changes occur in a large share of Onion and Potato districts.

## 6. R4 diagnostic (diagnostic only)

In [10]:
r4 = man[man["item"] == "r4_diagnostic"][["name", "value"]].rename(columns={"name": "crop"})
season_level = pd.read_csv(OUT / "audit" / "yield_consistency.csv")
display(r4)
season_level

,crop,value
19,Onion,rows_computable=7898; max_abs_rel_diff=0.010000000000000064; design window only; no flag
20,Potato,rows_computable=8862; max_abs_rel_diff=0.06944444444444436; design window only; no flag
21,Wheat,rows_computable=10659; max_abs_rel_diff=0.010500000000000016; design window only; no flag


,crop,rows_computable,max_abs_rel_diff
0,Onion,11813,0.010000
1,Potato,11389,0.069444
2,Wheat,10694,0.010500


**R4 is diagnostic only.** It reports the number of rows where the check can be made (area > 0 and production > 0, non-null), and the largest relative difference between the published yield and production ÷ area. This covers the design window, at the annual grain (manifest) and the season grain (audit). **No row is flagged, nothing is excluded, and no tolerance is active.** The 1% cut-off of revision 1 was withdrawn.

## 7. R13: provider query pending (no cause asserted)

In [11]:
r13 = design[design["pq_reasons"].str.contains("provider_query_pending")]
print("R13 rows:", len(r13), r13.groupby("apy_state").size().to_dict())
r13[["crop", "apy_state", "apy_district", "source_apy_year", "area_ha", "production_tonnes", "yield_t_per_ha",
     "annual_comparability", "pq_class", "pq_reasons"]].head(10)

R13 rows: 37 {'Gujarat': 19, 'West Bengal': 18}


,crop,apy_state,apy_district,source_apy_year,area_ha,production_tonnes,yield_t_per_ha,annual_comparability,pq_class,pq_reasons
9835,Potato,Gujarat,Ahmedabad,1997-1998,9300.0,50048.0,5.382,whole_year_row,review,provider_query_pending
9845,Potato,Gujarat,Amreli,1997-1998,6700.0,36.0,0.005,whole_year_row,review,provider_query_pending
9869,Potato,Gujarat,Banas Kantha,1997-1998,55200.0,198983.0,3.605,whole_year_row,review,provider_query_pending
9891,Potato,Gujarat,Bharuch,1997-1998,134000.0,706.0,0.005,whole_year_row,review,provider_query_pending
9893,Potato,Gujarat,Bhavnagar,1997-1998,12900.0,87.0,0.007,whole_year_row,review,provider_query_pending
9897,Potato,Gujarat,Dangs,1997-1998,8800.0,53.0,0.006,whole_year_row,review,provider_query_pending
9904,Potato,Gujarat,Gandhinagar,1997-1998,1900.0,18107.0,9.530,whole_year_row,review,provider_query_pending
9926,Potato,Gujarat,Jamnagar,1997-1998,18200.0,1521.0,0.084,whole_year_row,review,provider_query_pending
9944,Potato,Gujarat,Junagadh,1997-1998,12000.0,86.0,0.007,whole_year_row,review,provider_query_pending
9946,Potato,Gujarat,Kachchh,1997-1998,99000.0,6707.0,0.068,whole_year_row,review,provider_query_pending


In [12]:
sums = pd.read_csv(OUT / "audit" / "state_year_sums.csv")
sums[(sums["crop"] == "Potato") & sums["state"].isin(["West Bengal", "Gujarat"])
     & sums["apy_year"].isin(["1997-1998", "1998-1999"])]

,crop,state,apy_year,area_ha,production_tonnes
507,Potato,Gujarat,1997-1998,797400.0,629027.0
508,Potato,Gujarat,1998-1999,869368.0,1318379.0
877,Potato,West Bengal,1997-1998,19616403.0,19732232.0
878,Potato,West Bengal,1998-1999,318199.0,6689636.0


**R13 (D-48, Option 1)** covers **all** annual rows of West Bengal and Gujarat Potato for 1997-98: 18 + 19 rows.
- These rows are `review`. They are **not excluded** and their values are **unchanged**.
- **No cause is asserted.** In particular, this does not say the values are a unit error. The West Bengal district areas sum far above the 1998-99 total, and several Gujarat yields are very low, but only DES can explain them.
- The DES enquiry is drafted in the plan (§15) and **has not been sent**. R13 stays until an answer is recorded under a new decision.

## 8. Reserved years: counts only

In [13]:
cnt = man[man["item"] == "counts"].copy()
cnt[["crop", "source_apy_year", "pq_class"]] = cnt["name"].str.split("|", expand=True)
cnt["rows"] = cnt["value"].astype(int)
reserved = cnt[cnt["source_apy_year"].isin(pc.RESERVED)]
print("reserved rows in the product:", reserved["rows"].sum(), "| 2022-23 rows in the product:",
      int(cnt["source_apy_year"].eq("2022-2023").sum()))
reserved.pivot_table(index=["crop", "source_apy_year"], columns="pq_class", values="rows", aggfunc="sum", fill_value=0)

reserved rows in the product: 4163 | 2022-23 rows in the product: 0


pq_class                annotated   ok  review  unusable
crop   source_apy_year                                  
Onion  2019-2020              241  231       0         2
       2020-2021              225  164       1         0
       2021-2022              199  176       0         0
Potato 2019-2020              259  206       0         1
       2020-2021              280  147       1         0
       2021-2022              270  153       4         0
Wheat  2019-2020              517    1       0         2
       2020-2021              534    1       0         0
       2021-2022              547    1       0         0

These are **counts only**, from the output manifest. No reserved value was used to design or tune any rule, and none is displayed here. The values exist in `production_context_v1.csv`, to be read only by downstream protocols that are themselves approved. Any result that uses them must repeat the D-47 caveat on publication timing.

## 9. Limitations and downstream implications

**Limitations**
- **The two-year lag is an assumption.** No district-level release history has been verified, and secondary metadata suggests that recent years were published late. The input is one December 2023 vintage, so earlier years carry revised values, not first-published ones.
- **Blocked rules:** R5 needs official geographic areas, R6 needs reference yields, R7 needs a robust reference, R8b needs LGD or gazette split history, and the R4 row flag needs a justified tolerance. Until then, implausible areas or yields are visible only through R13 and the audit.
- **Geography:** history is re-labelled with present-day names (for example, Telangana before 2014). Split districts are not reconciled, which is why the trailing summary is deferred to v1.1.
- **Coverage:** APY annual values exist for only part of the mandi district-years (Phase 4: roughly 47% Onion, 62% Potato, 94% Wheat).

**Downstream (Phase 11/12, not started)**
- **Allowed:** a slow, annual prior for ranking or shortlisting source districts, joined only through the D-18 crosswalk, with `pq_class` and `annual_comparability` as explicit filters and sensitivity splits.
- **Not allowed:** treating production as marketable surplus, weekly supply or market capacity; any production-based cap (D-46); guessed coordinates or fuzzy joins.
- **Phase 12 is still blocked** by missing inputs: transport and storage costs, capacities, fees and demand.